## 3. Preprocessing

### a. Load Dataset

In [9]:
import pandas as pd

dataset = pd.read_csv('../data/processed/harga_gabungan.csv')
dataset.head()

,tanggal,wilayah,komoditas,harga
0,2019-01-01,Aceh,Bawang Merah Ukuran Sedang,29750.0
1,2019-02-01,Aceh,Bawang Merah Ukuran Sedang,29750.0
2,2019-03-01,Aceh,Bawang Merah Ukuran Sedang,29000.0
3,2019-04-01,Aceh,Bawang Merah Ukuran Sedang,37100.0
4,2019-05-01,Aceh,Bawang Merah Ukuran Sedang,37100.0


### b. Sort Data

In [10]:
dataset['tanggal'] = pd.to_datetime(dataset['tanggal'])
dataset = dataset.sort_values(['wilayah', 'komoditas', 'tanggal']).reset_index(drop=True)

dataset.head()

,tanggal,wilayah,komoditas,harga
0,2019-01-01,Aceh,Bawang Merah Ukuran Sedang,29750.0
1,2019-02-01,Aceh,Bawang Merah Ukuran Sedang,29750.0
2,2019-03-01,Aceh,Bawang Merah Ukuran Sedang,29000.0
3,2019-04-01,Aceh,Bawang Merah Ukuran Sedang,37100.0
4,2019-05-01,Aceh,Bawang Merah Ukuran Sedang,37100.0


### c. Handle Missing Value (ffill)

In [11]:
dataset['harga'] = dataset.groupby(['wilayah', 'komoditas'])['harga'].transform('ffill')

dataset.to_csv('../data/processed/harga_gabungan.csv', index=False)
print("Saved harga_gabungan.csv:", dataset.shape)

Saved harga_gabungan.csv: (18304, 4)


### d. Lags and Time

In [12]:
dataset['tahun'] = dataset['tanggal'].dt.year
dataset['bulan'] = dataset['tanggal'].dt.month
dataset['kuartal'] = dataset['tanggal'].dt.quarter

dataset['harga_lag1'] = dataset.groupby(['wilayah', 'komoditas'])['harga'].shift(1)
dataset['harga_lag2'] = dataset.groupby(['wilayah', 'komoditas'])['harga'].shift(2)
dataset['harga_lag3'] = dataset.groupby(['wilayah', 'komoditas'])['harga'].shift(3)

dataset['harga_rolling3'] = dataset.groupby(['wilayah', 'komoditas'])['harga'].transform(lambda x: x.shift(1).rolling(3).mean())

dataset = dataset.dropna().reset_index(drop=True)

print("Shape setelah Feature Engineering:", dataset.shape)

Shape setelah Feature Engineering: (17680, 11)


### e. Split Dataset

In [13]:
CUTOFF_DATE = '2024-11-01'

train_df = dataset[dataset['tanggal'] < CUTOFF_DATE].copy()
test_df  = dataset[dataset['tanggal'] >= CUTOFF_DATE].copy()

print("Periode Training :", train_df['tanggal'].min().date(), "→", train_df['tanggal'].max().date())
print("Periode Testing  :", test_df['tanggal'].min().date(), "→", test_df['tanggal'].max().date())
print("Shape Train:", train_df.shape)
print("Shape Test :", test_df.shape)

Periode Training : 2019-04-01 → 2024-10-01
Periode Testing  : 2024-11-01 → 2026-04-01
Shape Train: (13936, 11)
Shape Test : (3744, 11)


### f. Encoding 

In [14]:
from sklearn.preprocessing import LabelEncoder

le_wilayah = LabelEncoder()
le_komoditas = LabelEncoder()

train_df['wilayah_enc'] = le_wilayah.fit_transform(train_df['wilayah'])
train_df['komoditas_enc'] = le_komoditas.fit_transform(train_df['komoditas'])

test_df['wilayah_enc'] = le_wilayah.transform(test_df['wilayah'])
test_df['komoditas_enc'] = le_komoditas.transform(test_df['komoditas'])

# Pisahkan X dan y
FITUR = ['tahun', 'bulan', 'kuartal', 'wilayah_enc', 'komoditas_enc', 
         'harga_lag1', 'harga_lag2', 'harga_lag3', 'harga_rolling3']
TARGET = 'harga'

X_train = train_df[FITUR]
y_train = train_df[TARGET]
X_test = test_df[FITUR]
y_test = test_df[TARGET]

### g. Scalling

In [15]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FITUR)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=FITUR)

print("Scaling dengan RobustScaler selesai, siap disave!")

Scaling dengan RobustScaler selesai, siap disave!


### h. Save the Preprocessing Result

In [16]:
import joblib

X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(scaler, '../api/models/scaler.joblib')
joblib.dump(le_wilayah, '../api/models/le_wilayah.joblib')
joblib.dump(le_komoditas, '../api/models/le_komoditas.joblib')

['../api/models/le_komoditas.joblib']